SIB - P6
A Jupyter Notebook with the neural network training exercises (losses, optimizers and the training loop).

In [1]:
import numpy as np

from si.data.dataset import Dataset
from si.metrics.accuracy import accuracy
from si.model_selection.cross_validation import k_fold_cross_validation
from si.neural_networks.activation import ReLUActivation, SigmoidActivation
from si.neural_networks.layers import DenseLayer
from si.neural_networks.losses import MeanSquaredError, BinaryCrossEntropy
from si.neural_networks.neural_network import NeuralNetwork
from si.neural_networks.optimizers import SGD

## XOR com a `NeuralNetwork` (slide 14)

In [2]:
X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], dtype=float)
y = np.array([0, 1, 1, 0], dtype=float)
ds = Dataset(X, y)

def train_xor(seed=1, learning_rate=0.5, middle_activation=True):
    np.random.seed(seed)
    net = NeuralNetwork(epochs=5000, batch_size=4, optimizer=SGD, learning_rate=learning_rate,
                        loss=MeanSquaredError, metric=accuracy)
    net.add(DenseLayer(4, (2,)))
    if middle_activation:
        net.add(SigmoidActivation())
    net.add(DenseLayer(1))
    net.add(SigmoidActivation())
    net.fit(ds)
    return net.predict(ds).round(2).ravel(), net.score(ds), net.history[5000]['loss']

for name, kwargs in [("original", {}),
                     ("sem a SigmoidActivation do meio", {"middle_activation": False}),
                     ("learning_rate=0.01", {"learning_rate": 0.01}),
                     ("seed(0)", {"seed": 0})]:
    pred, acc, loss = train_xor(**kwargs)
    print(f"{name:32s} previsões: {pred} | accuracy: {acc:.2f} | loss final: {loss:.4f}")

original                         previsões: [0.05 0.95 0.95 0.07] | accuracy: 1.00 | loss final: 0.0030
sem a SigmoidActivation do meio  previsões: [0.5 0.5 0.5 0.5] | accuracy: 0.50 | loss final: 0.2500


learning_rate=0.01               previsões: [0.5 0.5 0.5 0.5] | accuracy: 0.50 | loss final: 0.2499
seed(0)                          previsões: [0.5 0.5 0.5 0.5] | accuracy: 0.50 | loss final: 0.2500


**O que se observa**

- **Sem a sigmoide do meio:** as duas camadas densas colapsam numa só transformação linear. Um modelo linear não separa a XOR, por isso fica nas previsões de 0.5.
- **`learning_rate=0.01`:** os passos são demasiado pequenos para sair da zona inicial em 5000 épocas, por isso ainda não aprendeu.
- **`seed(0)`:** a arquitetura e os dados são os mesmos, mas partindo de outros pesos iniciais o treino fica preso. A perda de uma rede não é convexa, por isso o resultado depende da inicialização.

## Exercício 16 – construir, treinar e avaliar uma rede

O enunciado só fixa as 32 features e a tarefa binária, por isso uso um dataset sintético com sinal linear (1000 amostras). Com rótulos puramente aleatórios a accuracy esperada seria 0.5 e não diria nada sobre a rede.

Arquitetura: 32 → 16 → 8 → 1 (as camadas densas reduzem as unidades a metade, exceto a última).

In [3]:
np.random.seed(42)
X = np.random.randn(1000, 32)
true_weights = np.random.randn(32)
y = (X.dot(true_weights) + np.random.randn(1000) > 0).astype(float)
dataset = Dataset(X, y)

model = NeuralNetwork(epochs=100, batch_size=16, optimizer=SGD, learning_rate=0.01,
                      loss=BinaryCrossEntropy, metric=accuracy)
model.add(DenseLayer(16, (32,)))
model.add(ReLUActivation())
model.add(DenseLayer(8))
model.add(ReLUActivation())
model.add(DenseLayer(1))
model.add(SigmoidActivation())

scores = k_fold_cross_validation(model, dataset, cv=5, seed=42)
print("Accuracy por fold:", np.round(scores, 3))
print(f"Média: {np.mean(scores):.3f} (+/- {np.std(scores):.3f})")

Accuracy por fold: [0.915 0.94  0.925 0.89  0.935]
Média: 0.921 (+/- 0.018)


### Exercício 16 com `Dataset.from_random`

O mesmo modelo com o gerador de dados do próprio pacote. Aqui os rótulos são aleatórios e independentes das features, por isso o resultado esperado é cerca de 0.5: não há nada para aprender. Serve para confirmar que a rede corre e que o k-fold não dá valores artificialmente altos.

In [4]:
np.random.seed(42)
random_dataset = Dataset.from_random(1000, 32, 2)

model = NeuralNetwork(epochs=100, batch_size=16, optimizer=SGD, learning_rate=0.01,
                      loss=BinaryCrossEntropy, metric=accuracy)
model.add(DenseLayer(16, (32,)))
model.add(ReLUActivation())
model.add(DenseLayer(8))
model.add(ReLUActivation())
model.add(DenseLayer(1))
model.add(SigmoidActivation())

scores = k_fold_cross_validation(model, random_dataset, cv=5, seed=42)
print("Accuracy por fold:", np.round(scores, 3))
print(f"Média: {np.mean(scores):.3f} (+/- {np.std(scores):.3f})")

Accuracy por fold: [0.525 0.44  0.44  0.53  0.45 ]
Média: 0.477 (+/- 0.041)
